### Node Features Range Distribution

In [14]:
import pandas as pd
from rdkit import Chem
from collections import defaultdict

# Load dataset
df = pd.read_csv('C:/Users/suman/OneDrive/Bureau/Internship_Study/GNN_On_OdorPrediction/data/OdorSmiles_Updated.csv', encoding='ISO-8859-1')
smiles_list = df['SMILES'].dropna().tolist()

# Initialize feature distributions
feature_distribution = {
    'atomic_num': defaultdict(int),
    'degree': defaultdict(int),
    'formal_charge': defaultdict(int),
    'num_hs': defaultdict(int),
    'num_radical_electrons': defaultdict(int),
    'valence': defaultdict(int),
    'smallest_ring': defaultdict(int),
}

for smiles in smiles_list:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        continue

    for atom in mol.GetAtoms():
        feature_distribution['atomic_num'][atom.GetAtomicNum()] += 1
        feature_distribution['degree'][atom.GetDegree()] += 1
        feature_distribution['formal_charge'][atom.GetFormalCharge()] += 1
        feature_distribution['num_hs'][atom.GetTotalNumHs()] += 1
        feature_distribution['num_radical_electrons'][atom.GetNumRadicalElectrons()] += 1
        feature_distribution['valence'][atom.GetTotalValence()] += 1
        feature_distribution['smallest_ring'][atom.GetOwningMol().GetRingInfo().NumAtomRings(atom.GetIdx())] += 1

# Summary
summary = {}
for key, dist in feature_distribution.items():
    summary[key] = {
        "min": min(dist.keys()) if dist else None,
        "max": max(dist.keys()) if dist else None,
        "most_common": max(dist.items(), key=lambda x: x[1]) if dist else None,
        "total_unique_values": len(dist)
    }

# Print results
import pprint
pprint.pprint(summary)


[11:37:05] WARNING: not removing hydrogen atom without neighbors
[11:37:05] WARNING: not removing hydrogen atom without neighbors


{'atomic_num': {'max': 35,
                'min': 1,
                'most_common': (6, 37973),
                'total_unique_values': 13},
 'degree': {'max': 4,
            'min': 0,
            'most_common': (2, 23287),
            'total_unique_values': 5},
 'formal_charge': {'max': 2,
                   'min': -2,
                   'most_common': (0, 45032),
                   'total_unique_values': 5},
 'num_hs': {'max': 4,
            'min': 0,
            'most_common': (2, 12575),
            'total_unique_values': 5},
 'num_radical_electrons': {'max': 1,
                           'min': 0,
                           'most_common': (0, 45106),
                           'total_unique_values': 2},
 'smallest_ring': {'max': 4,
                   'min': 0,
                   'most_common': (0, 28927),
                   'total_unique_values': 5},
 'valence': {'max': 6,
             'min': 0,
             'most_common': (4, 37993),
             'total_unique_values': 7}}


In [15]:
import pandas as pd
from rdkit import Chem
from collections import defaultdict
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_csv('C:/Users/suman/OneDrive/Bureau/Internship_Study/GNN_On_OdorPrediction/data/OdorSmiles_Updated.csv', encoding='ISO-8859-1')
smiles_list = df['SMILES'].dropna().tolist()

# Initialize feature distributions
feature_distribution = {
    'atomic_num': defaultdict(int),
    'degree': defaultdict(int),
    'formal_charge': defaultdict(int),
    'num_hs': defaultdict(int),
    'num_radical_electrons': defaultdict(int),
    'valence': defaultdict(int),
    'smallest_ring': defaultdict(int),
}

for smiles in smiles_list:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        continue

    for atom in mol.GetAtoms():
        feature_distribution['atomic_num'][atom.GetAtomicNum()] += 1
        feature_distribution['degree'][atom.GetDegree()] += 1
        feature_distribution['formal_charge'][atom.GetFormalCharge()] += 1
        feature_distribution['num_hs'][atom.GetTotalNumHs()] += 1
        feature_distribution['num_radical_electrons'][atom.GetNumRadicalElectrons()] += 1
        feature_distribution['valence'][atom.GetTotalValence()] += 1
        feature_distribution['smallest_ring'][mol.GetRingInfo().NumAtomRings(atom.GetIdx())] += 1

expanded_data = {}
for feature, dist in feature_distribution.items():
    values = []
    for val, count in dist.items():
        values.extend([val] * count)
    expanded_data[feature] = values

# Align all lists to same length by padding with NaN
max_len = max(len(lst) for lst in expanded_data.values())
for feature in expanded_data:
    if len(expanded_data[feature]) < max_len:
        expanded_data[feature] += [None] * (max_len - len(expanded_data[feature]))

# Create DataFrame
df_expanded = pd.DataFrame(expanded_data)

# Plotting function
def plot_feature_boxplots(dataframe, save_path="feature_distributions.png"):
    num_features = dataframe.shape[1]
    num_cols = 4
    num_rows = int(np.ceil(num_features / num_cols))

    plt.figure(figsize=(4 * num_cols, 4 * num_rows))

    for i, col in enumerate(dataframe.columns):
        ax = plt.subplot(num_rows, num_cols, i + 1)
        sns.countplot(x=dataframe[col], ax=ax, color='skyblue')
        min_val = dataframe[col].min()
        max_val = dataframe[col].max()
        ax.set_title(f"{col}\nmin={min_val}, max={max_val}", fontsize=9)
        ax.set_xlabel("")
        ax.set_ylabel("")

    plt.suptitle("Node Feature Distributions", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    plt.savefig(save_path, dpi=300)
    plt.close()

# Call the function
plot_feature_boxplots(df_expanded, save_path="atomic_feature_distributions.png")
print("Hist plot saved as 'atomic_feature_distributions.png'")

[11:37:09] WARNING: not removing hydrogen atom without neighbors
[11:37:09] WARNING: not removing hydrogen atom without neighbors


Hist plot saved as 'atomic_feature_distributions.png'


### Molecular Feature Distribution

In [20]:
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('C:/Users/suman/OneDrive/Bureau/Internship_Study/GNN_On_OdorPrediction/data/OdorSmiles_Updated.csv', encoding='ISO-8859-1')
smiles_list = df['SMILES'].dropna().tolist()

mol_feature_dict = {
    'molecular_weight': [],
    'logp': [],
    'tpsa': [],
    'num_rings': [],
    'num_rotatable_bonds': [],
    'num_H_bond_donors': [],
    'num_H_bond_acceptors': [],
    'heavy_atom_count': [],
    'formal_charge':[],
    'complexity': []
}

for smiles in smiles_list:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        continue

    try:
        mol_feature_dict['molecular_weight'].append(Descriptors.MolWt(mol))
        mol_feature_dict['logp'].append(Descriptors.MolLogP(mol))
        mol_feature_dict['tpsa'].append(rdMolDescriptors.CalcTPSA(mol))
        mol_feature_dict['num_rings'].append(rdMolDescriptors.CalcNumRings(mol))
        mol_feature_dict['num_rotatable_bonds'].append(Descriptors.NumRotatableBonds(mol))
        mol_feature_dict['num_H_bond_donors'].append(rdMolDescriptors.CalcNumHBD(mol))
        mol_feature_dict['num_H_bond_acceptors'].append(rdMolDescriptors.CalcNumHBA(mol))
        mol_feature_dict['heavy_atom_count'].append(Descriptors.HeavyAtomCount(mol))
        formal_charge = sum(atom.GetFormalCharge() for atom in mol.GetAtoms())
        mol_feature_dict['formal_charge'].append(formal_charge)
        mol_feature_dict['complexity'].append(Descriptors.FractionCSP3(mol))
    except:
        continue

# Create DataFrame
mol_df = pd.DataFrame(mol_feature_dict)

# Drop entirely empty columns (e.g., if all were skipped)
mol_df.dropna(axis=1, how='all', inplace=True)

# Plotting function
def plot_mol_feature_histograms(dataframe, save_path="mol_feature_histograms.png"):
    num_features = dataframe.shape[1]
    num_cols = 4
    num_rows = int(np.ceil(num_features / num_cols))

    plt.figure(figsize=(4 * num_cols, 4 * num_rows))

    for i, col in enumerate(dataframe.columns):
        data = dataframe[col].dropna()
        if data.empty:
            continue  # Skip empty

        ax = plt.subplot(num_rows, num_cols, i + 1)
        sns.histplot(data, ax=ax, bins=30, color='salmon')

        min_val = round(data.min(), 2)
        max_val = round(data.max(), 2)
        ax.set_title(f"{col}\nmin={min_val}, max={max_val}", fontsize=9)
        ax.set_xlabel("")
        ax.set_ylabel("")

    plt.suptitle("Molecular Feature Distributions", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    plt.savefig(save_path, dpi=300)
    plt.close()

# Call the function
plot_mol_feature_histograms(mol_df, save_path="mol_feature_histograms.png")
print("Histogram plot saved as 'mol_feature_histograms.png'")


[16:15:00] WARNING: not removing hydrogen atom without neighbors
[16:15:00] WARNING: not removing hydrogen atom without neighbors


Histogram plot saved as 'mol_feature_histograms.png'


In [ ]:
import pandas as pd
from rdkit import Chem
from torch_geometric.data import Data
from torch_geometric.utils import from_networkx
import networkx as nx

df = pd.read_csv('C:/Users/suman/OneDrive/Bureau/Internship_Study/GNN_On_OdorPrediction/data/OdorSmiles_Updated.csv', encoding='ISO-8859-1')

node_counts = []

for smiles in df['SMILES']:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        continue

    num_atoms = mol.GetNumAtoms()
    node_counts.append(num_atoms)

# Get the max
max_nodes = max(node_counts)
print(f"Maximum number of atoms (nodes) in your dataset: {max_nodes}")



[20:57:16] WARNING: not removing hydrogen atom without neighbors
[20:57:16] WARNING: not removing hydrogen atom without neighbors


Maximum number of atoms (nodes) in your dataset: 58
